# STimage-1K4M → HEST Pipeline
## Step 1: Overlap / Duplicate Detection (6-feature scoring)

Detects which STimage-1K4M samples already exist in HEST using **6 features**:

| # | STimage column | HEST column | Precision |
|---|---------------|-------------|-----------|
| 1 | `slide` → **GSM ID** | `download_page_link1` → GSM | Sample-level exact |
| 2 | `slide` → **GSE ID** | `download_page_link1` / `study_link` | Study-level exact |
| 3 | **`pmid`** | `study_link` → PMID | Study-level exact |
| 4 | **`title`** (paper title) | `dataset_title` | Fuzzy string match |
| 5 | **`tech`** (ST/Visium/VisiumHD) | `st_technology` | Categorical filter |
| 6 | **`species`** + **`tissue`** + **`spot_num`** | `species` + `tissue` + `spots_under_tissue` | Composite fingerprint |

Each feature contributes a weighted **confidence score**. Samples are labelled DEFINITE / LIKELY / POSSIBLE / NOVEL.

In [ ]:
import pandas as pd
import re
from pathlib import Path
from difflib import SequenceMatcher

# ── Paths ──────────────────────────────────────────────────────────────────
STIMAGE_META = Path('../../STimage-1K4M/meta/meta_all_gene02122025.csv')
HEST_META_HF = 'hf://datasets/MahmoodLab/hest/HEST_v1_3_0.csv'

st   = pd.read_csv(STIMAGE_META)
hest = pd.read_csv(HEST_META_HF)

print(f'STimage-1K4M : {len(st):,} samples  columns: {list(st.columns)}')
print(f'HEST         : {len(hest):,} samples  columns: {list(hest.columns)}')

## Features 1 & 2 — GSM and GSE IDs

In [ ]:
def extract_id(pattern, text):
    m = re.search(pattern, str(text))
    return m.group(1) if m else None

# STimage: both live in 'slide'  e.g. 'GSE144239_GSM4284316'
st['gsm_id'] = st['slide'].apply(lambda x: extract_id(r'(GSM\d+)', x))
st['gse_id'] = st['slide'].apply(lambda x: extract_id(r'(GSE\d+)', x))

# HEST: GSM/GSE in download_page_link1  e.g. acc=GSM9427181
hest['gsm_id'] = hest['download_page_link1'].apply(lambda x: extract_id(r'acc=(GSM\d+)', x))
hest['gse_id'] = (
    hest['download_page_link1'].apply(lambda x: extract_id(r'acc=(GSE\d+)', x))
    .fillna(hest['study_link'].apply(lambda x: extract_id(r'acc=(GSE\d+)', x)))
)

hest_gsm_set = set(hest['gsm_id'].dropna())
hest_gse_set = set(hest['gse_id'].dropna())

st['feat1_gsm'] = st['gsm_id'].isin(hest_gsm_set)
st['feat2_gse'] = st['gse_id'].isin(hest_gse_set)

print(f'STimage GSM coverage : {st["gsm_id"].notna().sum()} / {len(st)}')
print(f'STimage GSE coverage : {st["gse_id"].notna().sum()} / {len(st)}')
print(f'HEST    GSM coverage : {hest["gsm_id"].notna().sum()} / {len(hest)}')
print(f'HEST    GSE coverage : {hest["gse_id"].notna().sum()} / {len(hest)}')
print(f'\nFeat1 GSM hits : {st["feat1_gsm"].sum()}')
print(f'Feat2 GSE hits : {st["feat2_gse"].sum()}')

## Feature 3 — PMID

In [ ]:
# STimage pmid can be comma-separated (multiple papers per sample)
def pmid_set(val):
    if pd.isna(val): return set()
    return {p.strip() for p in str(val).split(',') if p.strip().isdigit()}

st['pmid_set'] = st['pmid'].apply(pmid_set)

# HEST: extract PMID from pubmed study_link
hest['pmid'] = hest['study_link'].apply(
    lambda x: extract_id(r'pubmed\.ncbi\.nlm\.nih\.gov/(\d+)', x)
)
hest_pmid_set = set(hest['pmid'].dropna())

st['feat3_pmid'] = st['pmid_set'].apply(lambda s: bool(s & hest_pmid_set))

shared_pmids = {p for s in st[st['feat3_pmid']]['pmid_set'] for p in s} & hest_pmid_set
print(f'HEST PMID coverage   : {hest["pmid"].notna().sum()} / {len(hest)}')
print(f'Feat3 PMID hits      : {st["feat3_pmid"].sum()}')
print(f'Shared PMIDs         : {sorted(shared_pmids)}')

## Feature 4 — Title fuzzy match

In [ ]:
# STimage 'title' = paper title(s); HEST 'dataset_title' = dataset name
# Different phrasing but overlapping vocabulary — use SequenceMatcher

def normalise(text):
    if pd.isna(text): return ''
    return re.sub(r'[^a-z0-9 ]', '', str(text).lower())

hest_titles_norm = hest['dataset_title'].apply(normalise).tolist()

TITLE_THRESHOLD = 0.6

def best_title_score(raw):
    s = normalise(raw)
    if not s: return 0.0
    return round(max(
        SequenceMatcher(None, s[:200], h[:200]).ratio()
        for h in hest_titles_norm
    ), 3)

print('Computing title similarity (~1-2 min)...')
st['feat4_title_score'] = st['title'].apply(best_title_score)
st['feat4_title']       = st['feat4_title_score'] >= TITLE_THRESHOLD

print(f'Feat4 title hits (>= {TITLE_THRESHOLD}): {st["feat4_title"].sum()}')
st[st['feat4_title']][['slide','title','feat4_title_score']].head(10)

## Feature 5 — Technology compatibility

In [ ]:
# Normalise tech labels between the two datasets
TECH_MAP = {
    'ST'      : 'Spatial Transcriptomics',
    'Visium'  : 'Visium',
    'VisiumHD': 'Visium HD',
}
st['tech_norm'] = st['tech'].map(TECH_MAP)

hest_tech_set = set(hest['st_technology'].dropna())
print('HEST technologies:', sorted(hest_tech_set))

# Used as a corroborating filter, not a standalone overlap signal
st['feat5_tech'] = st['tech_norm'].isin(hest_tech_set)
print('STimage tech in HEST:', st.groupby('tech')['feat5_tech'].sum().to_dict())

## Feature 6 — Composite fingerprint (species + tissue + spot count ±10%)

In [ ]:
SPECIES_MAP = {
    'human'              : 'Homo sapiens',
    'mouse'              : 'Mus musculus',
    'rattus norvegicus'  : 'Rattus norvegicus',
    'pig'                : 'Sus scrofa',
    'human & mouse'      : None,
    'fish'               : None,
    'plant'              : None,
    'frog'               : None,
    'ambystoma mexicanum': None,
}
st['species_norm']   = st['species'].str.lower().map(SPECIES_MAP)
st['tissue_norm']    = st['tissue'].str.lower().str.strip()
hest['tissue_norm']  = hest['tissue'].str.lower().str.strip()

SPOT_TOL = 0.10  # ±10%

# Build lookup: {(species, tissue): [(spot_count, hest_id), ...]}
fp_lookup = {}
for _, row in hest.iterrows():
    key = (row.get('species', ''), row.get('tissue_norm', ''))
    if pd.notna(row.get('spots_under_tissue')):
        fp_lookup.setdefault(key, []).append((int(row['spots_under_tissue']), row['id']))

def fingerprint_match(row):
    key = (row['species_norm'], row['tissue_norm'])
    if key not in fp_lookup or pd.isna(row['spot_num']): return False
    st_spots = int(row['spot_num'])
    return any(
        abs(st_spots - h) / max(st_spots, 1) <= SPOT_TOL
        for h, _ in fp_lookup[key]
    )

st['feat6_fingerprint'] = st.apply(fingerprint_match, axis=1)
print(f'Feat6 fingerprint hits (species+tissue+spot±10%): {st["feat6_fingerprint"].sum()}')

## Confidence scoring — all 6 features combined

In [ ]:
# Weights: higher = more specific evidence
WEIGHTS = {
    'feat1_gsm'        : 1.00,  # exact sample-level ID — definitive alone
    'feat2_gse'        : 0.60,  # same study
    'feat3_pmid'       : 0.60,  # same paper
    'feat4_title'      : 0.40,  # fuzzy title match
    'feat5_tech'       : 0.10,  # tech match (filter/corroboration only)
    'feat6_fingerprint': 0.30,  # species+tissue+spot fingerprint
}

st['confidence'] = sum(
    st[feat].astype(float) * w for feat, w in WEIGHTS.items()
)

# Thresholds
def overlap_label(score):
    if score >= 1.00: return 'DEFINITE'   # GSM match alone, or multiple strong signals
    if score >= 0.60: return 'LIKELY'     # GSE + PMID, or title + fingerprint
    if score >= 0.30: return 'POSSIBLE'   # weak corroboration — manual review
    return 'NOVEL'

st['overlap_label'] = st['confidence'].apply(overlap_label)

print('=== Label counts ===')
print(st['overlap_label'].value_counts().to_string())
print('\n=== Feature contribution ===')
for feat, w in WEIGHTS.items():
    print(f'  {feat:25s} (w={w:.2f}): {int(st[feat].sum()):4d} hits')

## Summary table

In [ ]:
print('=== Overlap by technology ===')
print(st.groupby(['overlap_label','tech']).size().unstack(fill_value=0).to_string())

total    = len(st)
definite = (st['overlap_label'] == 'DEFINITE').sum()
likely   = (st['overlap_label'] == 'LIKELY').sum()
possible = (st['overlap_label'] == 'POSSIBLE').sum()
novel    = (st['overlap_label'] == 'NOVEL').sum()

print(f'\nTotal STimage samples : {total:,}')
print(f'DEFINITE overlap      : {definite:,}  → skip')
print(f'LIKELY overlap        : {likely:,}   → skip (spot-check a few)')
print(f'POSSIBLE              : {possible:,}  → manual review')
print(f'NOVEL                 : {novel:,}  → convert to HEST')

In [ ]:
# Save all outputs
out_dir = Path('../pipeline_output')
out_dir.mkdir(exist_ok=True)

feat_cols = [
    'slide','gsm_id','gse_id','pmid','tech','tissue','species','spot_num','gene_num',
    'feat1_gsm','feat2_gse','feat3_pmid','feat4_title','feat4_title_score',
    'feat5_tech','feat6_fingerprint','confidence','overlap_label'
]

st[feat_cols].to_csv(out_dir / 'stimage_overlap_scored.csv', index=False)
st[st['overlap_label'] == 'NOVEL'][feat_cols].to_csv(out_dir / 'stimage_novel.csv', index=False)
st[st['overlap_label'] == 'POSSIBLE'][feat_cols].to_csv(out_dir / 'stimage_manual_review.csv', index=False)
st[st['overlap_label'].isin(['DEFINITE','LIKELY'])][feat_cols].to_csv(out_dir / 'stimage_confirmed_overlap.csv', index=False)

print(f'Outputs in {out_dir}/')
print(f'  stimage_overlap_scored.csv      — full table with all 6 feature scores ({len(st)} rows)')
print(f'  stimage_novel.csv               — {novel} novel samples → convert to HEST')
print(f'  stimage_manual_review.csv       — {possible} samples → manual check')
print(f'  stimage_confirmed_overlap.csv   — {definite+likely} confirmed duplicates → skip')

## Manual review guide

For each sample in `stimage_manual_review.csv`:
1. **GSE number** → search https://www.ncbi.nlm.nih.gov/geo/
2. **Title** → verify paper matches a HEST study
3. **PMID** → cross-check at https://pubmed.ncbi.nlm.nih.gov/
4. If confirmed duplicate → move to `stimage_confirmed_overlap.csv`
5. If confirmed novel → move to `stimage_novel.csv`

## Next: Notebook 7 — Convert `stimage_novel.csv` samples to HEST format